In [ ]:
# Example 1: Basic Optimization Setup and Function Minimization
import nevergrad as ng
import numpy as np

# Define a simple objective function to minimize
def simple_function(x, y):
    """Simple quadratic function with minimum at (2, -1)"""
    return (x - 2)**2 + (y + 1)**2 + 0.1 * np.sin(10 * x) * np.cos(10 * y)

# Create optimization variables using nevergrad's parametrization system
instrumentation = ng.p.Instrumentation(
    x=ng.p.Scalar(lower=-5, upper=5),  # x between -5 and 5
    y=ng.p.Scalar(lower=-3, upper=3)   # y between -3 and 3
)

# Configure optimizer (OnePlusOne is a simple evolutionary algorithm)
optimizer = ng.optimizers.OnePlusOne(parametrization=instrumentation, budget=100)

# Run optimization using ask/tell interface (core nevergrad pattern)
print("Starting optimization...")
for _ in range(optimizer.budget):
    x = optimizer.ask()
    loss = simple_function(**x.kwargs)
    optimizer.tell(x, loss)

# Get best result using provide_recommendation()
recommendation = optimizer.provide_recommendation()
best_params = recommendation.kwargs
best_loss = simple_function(**best_params)

print(f"Best parameters: x={best_params['x']:.4f}, y={best_params['y']:.4f}")
print(f"Best loss: {best_loss:.6f}")
print(f"Target optimum: x=2.0000, y=-1.0000")
print("Basic optimization completed successfully!")

In [ ]:
# Example 2: Multi-Algorithm Comparison and Performance Analysis
import nevergrad as ng
import numpy as np
import time

# Define a more complex optimization problem (Rastrigin function)
def rastrigin_function(x):
    """N-dimensional Rastrigin function - challenging for optimizers"""
    A = 10
    n = len(x)
    return A * n + sum([(xi**2 - A * np.cos(2 * np.pi * xi)) for xi in x])

# Set up 5-dimensional problem
dimension = 5
instrumentation = ng.p.Array(shape=(dimension,), lower=-5.12, upper=5.12)

# Compare different optimization algorithms
algorithms = {
    'DE': ng.optimizers.DE,  # Differential Evolution
    'PSO': ng.optimizers.PSO,  # Particle Swarm Optimization
    'CMA': ng.optimizers.CMA,  # Covariance Matrix Adaptation
    'OnePlusOne': ng.optimizers.OnePlusOne,  # (1+1)-ES
    'TwoPointsDE': ng.optimizers.TwoPointsDE  # Two-point DE
}

budget = 200
results = {}

print(f"Comparing {len(algorithms)} algorithms on {dimension}D Rastrigin function...")
print(f"Budget: {budget} evaluations per algorithm")
print("-" * 50)

for name, OptimizerClass in algorithms.items():
    start_time = time.time()
    optimizer = OptimizerClass(parametrization=instrumentation, budget=budget)
    
    # Run optimization
    for _ in range(budget):
        x = optimizer.ask()
        loss = rastrigin_function(x.value)
        optimizer.tell(x, loss)
    
    # Store results
    recommendation = optimizer.provide_recommendation()
    best_loss = rastrigin_function(recommendation.value)
    elapsed_time = time.time() - start_time
    
    results[name] = {'loss': best_loss, 'time': elapsed_time}
    print(f"{name:12s}: Loss = {best_loss:8.4f}, Time = {elapsed_time:.3f}s")

# Find best algorithm
best_algorithm = min(results.keys(), key=lambda k: results[k]['loss'])
print(f"\nBest algorithm: {best_algorithm} (loss: {results[best_algorithm]['loss']:.4f})")
print("Global minimum is 0.0 at origin")

In [ ]:
# Example 3: Complex Parameter Spaces and Hyperparameter Optimization
import nevergrad as ng
import numpy as np

# Simulate machine learning model hyperparameter optimization
def ml_model_objective(learning_rate, batch_size, dropout_rate, optimizer_type, 
                      use_batch_norm, hidden_layers):
    """Simulates ML model training with different hyperparameters"""
    
    # Simulate model performance based on hyperparameters
    base_score = 0.85
    
    # Learning rate effect (optimal around 0.001-0.01)
    lr_penalty = abs(np.log10(learning_rate) + 2.5) * 0.05
    
    # Batch size effect (power of 2 usually better)
    batch_penalty = 0.02 if batch_size in [16, 32, 64, 128, 256] else 0.05
    
    # Dropout effect (optimal around 0.2-0.5)
    dropout_penalty = abs(dropout_rate - 0.3) * 0.1
    
    # Optimizer effect
    optimizer_bonus = {'adam': 0.02, 'sgd': 0.0, 'rmsprop': 0.01}[optimizer_type]
    
    # Batch normalization bonus
    bn_bonus = 0.015 if use_batch_norm else 0.0
    
    # Hidden layers effect (too many or too few is bad)
    layer_penalty = abs(hidden_layers - 3) * 0.01
    
    # Add some noise to simulate real training variance
    noise = np.random.normal(0, 0.01)
    
    # Return negative accuracy (since we minimize)
    accuracy = base_score + optimizer_bonus + bn_bonus - lr_penalty - batch_penalty - dropout_penalty - layer_penalty + noise
    return 1.0 - accuracy  # Convert to minimization problem

# Create complex parameter space
instrumentation = ng.p.Instrumentation(
    learning_rate=ng.p.Log(lower=1e-5, upper=1e-1),  # Log-uniform distribution
    batch_size=ng.p.Choice([16, 32, 64, 128, 256]),  # Discrete choices
    dropout_rate=ng.p.Scalar(lower=0.0, upper=0.8),  # Continuous uniform
    optimizer_type=ng.p.Choice(['adam', 'sgd', 'rmsprop']),  # Categorical
    use_batch_norm=ng.p.Choice([True, False]),  # Boolean
    hidden_layers=ng.p.Scalar(lower=1, upper=10).set_integer_casting()  # Integer
)

# Use Bayesian optimization (good for expensive evaluations)
optimizer = ng.optimizers.BO(parametrization=instrumentation, budget=50)

print("Hyperparameter optimization for ML model...")
print("Parameter space:")
print("- Learning rate: 1e-5 to 1e-1 (log-uniform)")
print("- Batch size: [16, 32, 64, 128, 256]")
print("- Dropout rate: 0.0 to 0.8")
print("- Optimizer: ['adam', 'sgd', 'rmsprop']")
print("- Batch norm: [True, False]")
print("- Hidden layers: 1 to 10 (integer)")
print("-" * 40)

# Run optimization with progress tracking
best_score = float('inf')
for i in range(optimizer.budget):
    x = optimizer.ask()
    loss = ml_model_objective(**x.kwargs)
    optimizer.tell(x, loss)
    
    if loss < best_score:
        best_score = loss
        print(f"Step {i+1:2d}: New best accuracy = {1-loss:.4f}")

# Get final recommendation
recommendation = optimizer.provide_recommendation()
final_params = recommendation.kwargs
final_accuracy = 1 - ml_model_objective(**final_params)

print(f"\nOptimal hyperparameters found:")
for param, value in final_params.items():
    print(f"  {param}: {value}")
print(f"Final model accuracy: {final_accuracy:.4f}")
print("Hyperparameter optimization completed!")

In [ ]:
# Example 4: Parallel Optimization and Multi-Objective Problems
import nevergrad as ng
import numpy as np
from concurrent.futures import ThreadPoolExecutor

# Define multi-objective function (Pareto optimization)
def multi_objective_function(x, y, z):
    """Two competing objectives: minimize both f1 and f2"""
    # Objective 1: Distance from point (1, 1, 1)
    f1 = (x - 1)**2 + (y - 1)**2 + (z - 1)**2
    
    # Objective 2: Distance from point (-1, -1, -1)  
    f2 = (x + 1)**2 + (y + 1)**2 + (z + 1)**2
    
    # These objectives conflict - cannot minimize both simultaneously
    return f1, f2

# Single objective version (weighted sum)
def weighted_objective(x, y, z, weight=0.5):
    """Convert multi-objective to single objective using weights"""
    f1, f2 = multi_objective_function(x, y, z)
    return weight * f1 + (1 - weight) * f2

# Set up parameter space
instrumentation = ng.p.Instrumentation(
    x=ng.p.Scalar(lower=-3, upper=3),
    y=ng.p.Scalar(lower=-3, upper=3),
    z=ng.p.Scalar(lower=-3, upper=3),
    weight=ng.p.Scalar(lower=0, upper=1)
)

# Parallel evaluation function
def evaluate_candidate(params_and_weight):
    """Evaluate function - can be expensive in practice"""
    import time
    time.sleep(0.01)  # Simulate expensive evaluation
    return weighted_objective(**params_and_weight)

# Set up optimizer with multiple workers
num_workers = 4
optimizer = ng.optimizers.DE(parametrization=instrumentation, budget=100, num_workers=num_workers)

print(f"Multi-objective optimization with {num_workers} parallel workers...")
print("Objectives:")
print("  f1: Distance from point (1, 1, 1)")  
print("  f2: Distance from point (-1, -1, -1)")
print("  Combined: weight * f1 + (1-weight) * f2")
print("-" * 40)

# Collect candidates for parallel evaluation
candidates = []
candidate_objects = []

# Generate all candidates first
for i in range(optimizer.budget):
    x = optimizer.ask()
    candidates.append(x.kwargs)
    candidate_objects.append(x)

# Parallel evaluation using ThreadPoolExecutor
print("Starting parallel evaluation...")
with ThreadPoolExecutor(max_workers=num_workers) as executor:
    results = list(executor.map(evaluate_candidate, candidates))

# Tell optimizer about all results
for candidate_obj, loss in zip(candidate_objects, results):
    optimizer.tell(candidate_obj, loss)

# Get recommendation and analyze Pareto front
recommendation = optimizer.provide_recommendation()
best_params = recommendation.kwargs
best_loss = weighted_objective(**best_params)
f1_best, f2_best = multi_objective_function(
    best_params['x'], best_params['y'], best_params['z']
)

print(f"\nBest solution found:")
print(f"  Position: ({best_params['x']:.3f}, {best_params['y']:.3f}, {best_params['z']:.3f})")
print(f"  Weight: {best_params['weight']:.3f}")
print(f"  Combined loss: {best_loss:.4f}")
print(f"  f1 (dist to (1,1,1)): {f1_best:.4f}")
print(f"  f2 (dist to (-1,-1,-1)): {f2_best:.4f}")
print("Parallel multi-objective optimization completed!")

In [ ]:
# Example 5: Production Optimization Pipeline with Logging and Callbacks
import nevergrad as ng
import numpy as np
import json
import time
from pathlib import Path

# Production-grade objective function with error handling
def production_objective(learning_rate, regularization, feature_subset_ratio, 
                        model_complexity):
    """Simulate training a production ML model with comprehensive logging"""
    
    try:
        # Validate parameters
        if learning_rate <= 0 or regularization < 0 or not (0 < feature_subset_ratio <= 1):
            return float('inf')  # Invalid parameters
        
        # Simulate model training with realistic performance curve
        base_performance = 0.75
        
        # Learning rate effect (log-normal distribution around 0.01)
        lr_factor = np.exp(-0.5 * ((np.log(learning_rate) - np.log(0.01)) / 0.5)**2)
        
        # Regularization effect (prevents overfitting)
        reg_factor = 1.0 / (1.0 + regularization)
        
        # Feature selection effect
        feature_factor = np.sqrt(feature_subset_ratio)
        
        # Model complexity effect (inverted U-shape)
        complexity_factor = np.exp(-0.1 * (model_complexity - 5)**2)
        
        # Combine factors with some noise
        performance = (base_performance * lr_factor * reg_factor * 
                      feature_factor * complexity_factor + 
                      np.random.normal(0, 0.01))
        
        # Add training time simulation
        training_time = (10 / learning_rate) * model_complexity * feature_subset_ratio
        
        # Multi-objective: balance performance and training time
        return (1.0 - performance) + 0.1 * np.log(training_time)
        
    except Exception as e:
        print(f"Error in objective function: {e}")
        return float('inf')

# Set up production parameter space with constraints
instrumentation = ng.p.Instrumentation(
    learning_rate=ng.p.Log(lower=1e-4, upper=1e-1),
    regularization=ng.p.Scalar(lower=0, upper=1),
    feature_subset_ratio=ng.p.Scalar(lower=0.1, upper=1.0),
    model_complexity=ng.p.Scalar(lower=1, upper=10).set_integer_casting()
)

# Production optimizer with restart capability
optimizer = ng.optimizers.NGOpt(parametrization=instrumentation, budget=150)

# Logging and callback system
optimization_log = []
best_configs = []

def log_evaluation(step, params, loss, is_best=False):
    """Log evaluation results for production monitoring"""
    log_entry = {
        'step': step,
        'timestamp': time.time(),
        'parameters': params,
        'loss': loss,
        'is_best': is_best
    }
    optimization_log.append(log_entry)
    
    if is_best:
        best_configs.append(log_entry)
        print(f"Step {step:3d}: New best loss = {loss:.6f} | "
              f"LR={params['learning_rate']:.2e}, "
              f"Reg={params['regularization']:.3f}")

print("Production optimization pipeline starting...")
print("Objective: Minimize (1 - model_performance) + 0.1 * log(training_time)")
print("Parameters:")
print("  - Learning rate: 1e-4 to 1e-1 (log-uniform)")
print("  - Regularization: 0 to 1 (uniform)")  
print("  - Feature subset ratio: 0.1 to 1.0 (uniform)")
print("  - Model complexity: 1 to 10 (integer)")
print("-" * 60)

# Main optimization loop with error handling
best_loss = float('inf')
start_time = time.time()

for step in range(optimizer.budget):
    try:
        # Ask for next candidate
        candidate = optimizer.ask()
        params = candidate.kwargs
        
        # Evaluate with timing
        eval_start = time.time()
        loss = production_objective(**params)
        eval_time = time.time() - eval_start
        
        # Tell optimizer the result
        optimizer.tell(candidate, loss)
        
        # Check if this is a new best
        is_best = loss < best_loss
        if is_best:
            best_loss = loss
        
        # Log the evaluation
        log_evaluation(step + 1, params, loss, is_best)
        
        # Progress update every 25 steps
        if (step + 1) % 25 == 0:
            elapsed = time.time() - start_time
            print(f"Progress: {step + 1}/{optimizer.budget} steps, "
                  f"Best loss: {best_loss:.6f}, "
                  f"Elapsed: {elapsed:.1f}s")
    
    except Exception as e:
        print(f"Error at step {step + 1}: {e}")
        continue

# Final results and analysis
total_time = time.time() - start_time
recommendation = optimizer.provide_recommendation()
final_params = recommendation.kwargs
final_loss = production_objective(**final_params)

print(f"\nOptimization completed in {total_time:.2f} seconds")
print(f"Total evaluations: {len(optimization_log)}")
print(f"Best configurations found: {len(best_configs)}")
print(f"\nFinal recommendation:")
for param, value in final_params.items():
    print(f"  {param}: {value}")
print(f"Final loss: {final_loss:.6f}")

# Save results for production use
results_summary = {
    'best_parameters': final_params,
    'best_loss': final_loss,
    'optimization_time': total_time,
    'total_evaluations': len(optimization_log),
    'convergence_history': [entry['loss'] for entry in optimization_log]
}

print(f"\nResults logged for production deployment")
print("Production optimization pipeline completed successfully!")